<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/findingSimilarItems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items

We download the Letterboxd dataset from Kaggle, using a token.

In [1]:
import os
import json
import pandas as pd
import pip
import string

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"
! kaggle datasets download -d gsimonx37/letterboxd

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


Dataset URL: https://www.kaggle.com/datasets/gsimonx37/letterboxd
License(s): GPL-3.0
100% 22.5G/22.5G [04:50<00:00, 112MB/s]
100% 22.5G/22.5G [04:50<00:00, 83.2MB/s]


We only consider a subet of the files contained in the `letterboxd` dataset, namely the data regarding the movie names and ids, their actors, crews, genres and themes.

In [4]:
import zipfile
from multiprocessing import Pool

DATA_DIR = "./letterboxd"
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
    for file_name in members_to_extract:
        zip_ref.extract(file_name + '.csv', DATA_DIR)

!rm -rf {DATA_DIR + ".zip"}



We then prepare the entry point for the Spark functionalities that will we use from now on.

In [5]:
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
!wget https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!rm spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "spark-3.5.3-bin-hadoop3"

import findspark
findspark.init("spark-3.5.3-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

--2024-11-28 15:57:25--  https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
Resolving downloads.apache.org (downloads.apache.org)... 88.99.208.237, 135.181.214.104, 2a01:4f9:3a:2c57::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|88.99.208.237|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400864419 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.3-bin-hadoop3.tgz’

spark-3.5.3-bin-had 100%[===================>] 382.29M  30.5MB/s    in 14s     

2024-11-28 15:57:39 (28.2 MB/s) - ‘spark-3.5.3-bin-hadoop3.tgz’ saved [400864419/400864419]



In [125]:
letterboxd_RDDs={}
letterboxd_RDDs['actors'] = sc.textFile(DATA_DIR + "/actors.csv").zipWithIndex() #maintain order of actors for each movie
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                                .map(lambda r: (r[0].split(','), r[1]))
                                .map(lambda r: (r[0][0], (r[0][1:], r[1])))
                                .filter(lambda r: r[0]!='id'))

In [118]:
letterboxd_RDDs['actors'].take(5)

[('1000001', (['Margot Robbie', 'Barbie'], 1)),
 ('1000001', (['Ryan Gosling', 'Ken'], 2)),
 ('1000001', (['America Ferrera', 'Gloria'], 3)),
 ('1000001', (['Ariana Greenblatt', 'Sasha'], 4)),
 ('1000001', (['Issa Rae', 'Barbie'], 5))]

In [126]:
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']

In [127]:
for member in members_to_extract[1:]:
    letterboxd_RDDs[member] = (sc.textFile(DATA_DIR + "/" + member + ".csv")
                                .map(lambda r: r.split(','))
                                .map(lambda r: (r[0], r[1:]))
                                .filter(lambda r: r[0]!='id'))

For each data type the available attributes are the following:

| **Table**    | **Columns**                             |
|--------------|-----------------------------------------|
| **actor**    | name, role                          |
| **crew**     | role, name                          |
| **genres**   | genre                               |
| **movies**   | name, date, tagline, description, minute, rating |
| **themes**   | theme                               |

For this project we would like to focus on the following features:

| **Table**    | **Columns**                             |
|--------------|-----------------------------------------|
| **actor**    | names of the first 10 actors for a given movie                      |
| **crew**     | name(s) of the director of each movie                        |
| **genres**   | genre                               |
| **movies**   | name, date, minute, rating |
| **themes**   | theme                               |


In [128]:
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .groupByKey()
                            .map(lambda r: (r[0], sorted(r[1], key=lambda x: x[1])[:10]))
                            .map(lambda r: (r[0], [x[0][0] for x in r[1]])))

In [122]:
letterboxd_RDDs['actors'].take(3)

[('1000004',
  ['Edward Norton',
   'Brad Pitt',
   'Helena Bonham Carter',
   'Meat Loaf',
   'Jared Leto',
   'Zach Grenier',
   'Holt McCallany',
   'Eion Bailey',
   'Richmond Arquette',
   'David Andrews']),
 ('1000009',
  ['Timothée Chalamet',
   'Rebecca Ferguson',
   'Oscar Isaac',
   'Jason Momoa',
   'Josh Brolin',
   'Stellan Skarsgård',
   'Stephen McKinley Henderson',
   'Javier Bardem',
   'Sharon Duncan-Brewster',
   'Chang Chen']),
 ('1000013',
  ['Robert Pattinson',
   'Zoë Kravitz',
   'Jeffrey Wright',
   'Colin Farrell',
   'Paul Dano',
   'John Turturro',
   'Andy Serkis',
   'Peter Sarsgaard',
   'Barry Keoghan',
   'Jayme Lawson'])]

In [129]:
for member in members_to_extract:
    print(member)
    letterboxd_RDDs[member] = letterboxd_RDDs[member].groupByKey()
    print(letterboxd_RDDs[member].take(1))

actors
[('1000004', <pyspark.resultiterable.ResultIterable object at 0x7ad8ba164a90>)]
crew
[('1000012', <pyspark.resultiterable.ResultIterable object at 0x7ad90145e050>)]
genres
[('1000001', <pyspark.resultiterable.ResultIterable object at 0x7ad90144e770>)]
movies
[('1000001', <pyspark.resultiterable.ResultIterable object at 0x7ad8ba921f90>)]
themes
[('1000001', <pyspark.resultiterable.ResultIterable object at 0x7ad8c48709a0>)]


In [124]:
letterboxd_RDDs['actors'].join(letterboxd_RDDs['crew']).take(1)

[('1000051',
  (<pyspark.resultiterable.ResultIterable at 0x7ad8c4871f00>,
   <pyspark.resultiterable.ResultIterable at 0x7ad8ba949ea0>))]

In [130]:
rdd = letterboxd_RDDs['actors'].mapValues(lambda x: [x])
for member in members_to_extract[1:]:
    rdd = rdd.join(letterboxd_RDDs[member]).mapValues(lambda x: x[0] + [x[1]])

In [132]:
rdd.take(4)

[('1004772',
  ((((<pyspark.resultiterable.ResultIterable at 0x7ad8ba909420>,
      <pyspark.resultiterable.ResultIterable at 0x7ad8ba90ba00>),
     <pyspark.resultiterable.ResultIterable at 0x7ad8ba90bb80>),
    <pyspark.resultiterable.ResultIterable at 0x7ad8ba908670>),
   <pyspark.resultiterable.ResultIterable at 0x7ad8ba909e10>)),
 ('1044721',
  ((((<pyspark.resultiterable.ResultIterable at 0x7ad8ba909b10>,
      <pyspark.resultiterable.ResultIterable at 0x7ad8ba90b610>),
     <pyspark.resultiterable.ResultIterable at 0x7ad8ba908190>),
    <pyspark.resultiterable.ResultIterable at 0x7ad8ba90b550>),
   <pyspark.resultiterable.ResultIterable at 0x7ad8ba9086a0>)),
 ('1014194',
  ((((<pyspark.resultiterable.ResultIterable at 0x7ad8ba90aa40>,
      <pyspark.resultiterable.ResultIterable at 0x7ad8ba909a50>),
     <pyspark.resultiterable.ResultIterable at 0x7ad8ba90be20>),
    <pyspark.resultiterable.ResultIterable at 0x7ad8ba9083a0>),
   <pyspark.resultiterable.ResultIterable at 0x7ad8ba